<img src="images/logo.jpg" style="display: block; margin-left: auto; margin-right: auto;" alt="לוגו הקורס">

# <p style="text-align: right; direction: rtl; float: right;">SQL מתקדם - UPDATE, DELETE, GROUP BY, JOIN</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
במחברת הקודמת למדנו ליצור טבלאות, להוסיף מידע ולשלוף אותו.<br>
עכשיו נלמד לשנות מידע קיים (<code>UPDATE</code>), למחוק (<code>DELETE</code>), לסכם לפי קבוצות (<code>GROUP BY</code> ו-<code>HAVING</code>)
ולחבר בין שתי טבלאות (<code>JOIN</code>).<br>
נעבוד עם נתונים של חברה: טבלת עובדים וטבלת מחלקות.
</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/tip.png" style="height: 50px !important;" alt="טיפ">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>חשוב להבין</strong>:<br>
            אם קיבלתם שגיאה (למשל <code>no such table</code> או <code>Cannot operate on a closed database</code>) -
הריצו מחדש מהתא <strong>"הכנת מסד הנתונים של החברה"</strong> ומטה. התא הזה בונה את הטבלאות מחדש, כך שתמיד מתחילים מנתונים נקיים.
        </p>
    </div>
</div>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">הכנת מסד הנתונים של החברה</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
לפני שמתחילים צריך נתונים. התא הבא יוצר מסד נתונים בשם <code>company.db</code> עם שתי טבלאות:<br>
🔹 <code>employees</code> - עובדים: מזהה, שם, מחלקה, שכר וגיל<br>
🔹 <code>departments</code> - מחלקות: מזהה, שם, מנהל וקומה<br><br>
הפקודה <code>DROP TABLE IF EXISTS</code> מוחקת טבלה ישנה אם קיימת, ולכן אפשר להריץ את התא כמה פעמים שרוצים - בכל פעם נקבל נתונים נקיים.
</p>

In [ ]:
import sqlite3

# אם כבר התחברנו בעבר (למשל הרצתם את התא פעם שנייה) - סוגרים את החיבור הישן
try:
    conn.close()
except NameError:
    pass

conn = sqlite3.connect('company.db')
cursor = conn.cursor()

# מוחקים טבלאות ישנות ובונים מחדש
cursor.execute('DROP TABLE IF EXISTS employees')
cursor.execute('DROP TABLE IF EXISTS departments')

cursor.execute('''
CREATE TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    department TEXT,
    salary REAL,
    age INTEGER
)
''')

cursor.execute('''
CREATE TABLE departments (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    manager TEXT,
    floor INTEGER
)
''')

employees = [
    (1, 'דני כהן', 'הנדסה', 15000, 34),
    (2, 'שרה לוי', 'שיווק', 12000, 29),
    (3, 'יוסי מזרחי', 'מכירות', 13500, 41),
    (4, 'רחל אברהם', 'הנדסה', 16000, 38),
    (5, 'משה דוד', 'כספים', 14000, 45),
    (6, 'מיכל שמעון', 'משאבי אנוש', 11000, 27),
    (7, 'אבי יעקב', 'מכירות', 13000, 52),
    (8, 'תמר ישראל', 'שיווק', 12500, 31),
    (9, 'עמית בן דוד', 'הנדסה', 15500, 36),
    (10, 'נועה ברק', 'כספים', 13800, 33),
    (11, 'אלי גולן', 'הנדסה', 17000, 67),
    (12, 'ליאת שרון', 'מחקר', 14500, 30),
]

departments = [
    (1, 'הנדסה', 'רחל אברהם', 3),
    (2, 'שיווק', 'שרה לוי', 2),
    (3, 'מכירות', 'יוסי מזרחי', 1),
    (4, 'כספים', 'משה דוד', 2),
    (5, 'משאבי אנוש', 'מיכל שמעון', 1),
    (6, 'רכש', 'גלית מור', 4),
]

cursor.executemany('INSERT INTO employees VALUES (?, ?, ?, ?, ?)', employees)
cursor.executemany('INSERT INTO departments VALUES (?, ?, ?, ?)', departments)
conn.commit()

print(f"נוצרו {len(employees)} עובדים ו-{len(departments)} מחלקות ✅")

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה כאן?</strong><br>
🔹 יצרנו שתי טבלאות והכנסנו אליהן נתונים עם <code>executemany</code><br>
🔹 <code>INSERT INTO employees VALUES (?, ?, ?, ?, ?)</code> - כשמכניסים ערך לכל העמודות, לפי הסדר, אפשר לא לרשום את שמות העמודות<br>
🔹 שימו לב לשני "מלכודות" שהכנו בכוונה: <strong>ליאת שרון</strong> עובדת במחלקת "מחקר" שאין בטבלת המחלקות, ובמחלקת <strong>רכש</strong> אין אף עובד. נשתמש בהן כשנגיע ל-JOIN
</p>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">פונקציית עזר: show</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
כדי שלא נכתוב את אותן שורות שוב ושוב, נכתוב פונקציה קטנה שמריצה שאילתה ומדפיסה כל שורה בשורה נפרדת.<br>
אנחנו נשתמש בה הרבה כדי להסתכל על הנתונים "לפני" ו"אחרי" כל שינוי.
</p>

In [ ]:
def show(query, params=()):
    """מריצה שאילתה SELECT ומדפיסה את התוצאות, שורה בכל שורה"""
    cursor.execute(query, params)
    rows = cursor.fetchall()
    for row in rows:
        print(row)
    if not rows:
        print("(אין תוצאות)")


show('SELECT * FROM employees LIMIT 4')

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה כאן?</strong><br>
הפונקציה מקבלת את טקסט השאילתה, ואופציונלית גם tuple של פרמטרים (לשימוש עם <code>?</code>).<br>
הדפסנו את ארבעת העובדים הראשונים. כל שורה היא tuple: (מזהה, שם, מחלקה, שכר, גיל).
</p>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">שאילתות עם פרמטרים - תזכורת</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
כשהערך שאנחנו מחפשים נמצא במשתנה, תמיד נעביר אותו עם <code>?</code> ולא עם f-string.<br>
נחפש את כל העובדים ששכרם גבוה מסכום מסוים:
</p>

In [ ]:
min_salary = 14000  # שנו כאן

show('SELECT name, department, salary FROM employees WHERE salary >= ?', (min_salary,))

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה כאן?</strong><br>
ה-<code>?</code> הוחלף בערך של <code>min_salary</code>. כדי לשנות את הסינון די לשנות את המשתנה.
</p>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">מיון והגבלה - ORDER BY ו-LIMIT</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
נרצה לדעת מי חמשת העובדים עם השכר הגבוה ביותר. קודם <strong>ממיינים</strong> לפי שכר מהגבוה לנמוך (<code>DESC</code>),
ואז <strong>מגבילים</strong> ל-5 שורות עם <code>LIMIT 5</code>:
</p>

In [ ]:
print("🏆 חמשת השכרים הגבוהים:")
show('SELECT name, salary FROM employees ORDER BY salary DESC LIMIT 5')

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה כאן?</strong><br>
הסדר חשוב: SQL קודם ממיין את כל העובדים ואז חותך את ה-5 הראשונים. אפשר למיין גם לפי שתי עמודות, למשל <code>ORDER BY department, salary DESC</code> (קודם לפי מחלקה, ובתוך כל מחלקה לפי שכר).
</p>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">חלק 1: עדכון נתונים - UPDATE</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
הפקודה <code>UPDATE</code> משנה ערכים בשורות שכבר קיימות בטבלה.<br>
המבנה: <code>UPDATE שם_טבלה SET עמודה = ערך WHERE תנאי</code>.<br><br>
נתנו העלאה של 10% לכל עובדי ההנדסה. נסתכל קודם על השכר <strong>לפני</strong>, נעדכן, ונסתכל שוב <strong>אחרי</strong>:
</p>

In [ ]:
print("לפני העדכון:")
show("SELECT name, salary FROM employees WHERE department = 'הנדסה'")

cursor.execute("""
    UPDATE employees
    SET salary = ROUND(salary * 1.1)
    WHERE department = 'הנדסה'
""")
print(f"\n✅ עודכנו {cursor.rowcount} עובדים")
conn.commit()

print("\nאחרי העדכון:")
show("SELECT name, salary FROM employees WHERE department = 'הנדסה'")

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה כאן?</strong><br>
🔹 <code>SET salary = ROUND(salary * 1.1)</code> - השכר החדש הוא השכר הישן כפול 1.1 (ו-<code>ROUND</code> מעגל למספר שלם)<br>
🔹 <code>cursor.rowcount</code> אומר לנו כמה שורות הושפעו. קיבלנו 4, כמספר עובדי ההנדסה<br>
🔹 בלי <code>conn.commit()</code> השינוי לא נשמר באופן סופי<br>
🔹 תמיד כדאי להסתכל "לפני" ו"אחרי" - כך מוודאים שעדכנו בדיוק את מי שרצינו
</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
גם ב-UPDATE אפשר (וצריך!) להשתמש ב-<code>?</code>. נעדכן שכר של עובד ספציפי לפי שם:
</p>

In [ ]:
new_salary = 15000   # שנו כאן
employee_name = 'משה דוד'

cursor.execute('UPDATE employees SET salary = ? WHERE name = ?', (new_salary, employee_name))
conn.commit()

print(f"עודכנו {cursor.rowcount} שורות")
show('SELECT name, salary FROM employees WHERE name = ?', (employee_name,))

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/tip.png" style="height: 50px !important;" alt="טיפ">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>שימו לב</strong>:<br>
            <strong>תמיד WHERE!</strong> אם שוכחים את ה-<code>WHERE</code> ב-UPDATE (או ב-DELETE), הפקודה חלה על <strong>כל</strong> השורות בטבלה.
בואו נראה בדיוק מה קורה - ואיך אפשר להתאושש, כל עוד לא עשינו <code>commit</code>.
        </p>
    </div>
</div>

In [ ]:
# ניסוי "מסוכן": שכחנו את ה-WHERE
cursor.execute('UPDATE employees SET salary = 0')
print(f"עודכנו {cursor.rowcount} שורות (כל הטבלה!)")

print("\nמה יש בטבלה עכשיו?")
show('SELECT name, salary FROM employees LIMIT 3')

# למזלנו עוד לא עשינו commit, אז אפשר לבטל את השינוי:
conn.rollback()

print("\nאחרי rollback:")
show('SELECT name, salary FROM employees LIMIT 3')

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה כאן?</strong><br>
🔹 בלי <code>WHERE</code> כל 12 העובדים קיבלו שכר 0!<br>
🔹 <code>conn.rollback()</code> מבטל את כל מה שנעשה מאז ה-<code>commit</code> האחרון, והשכר חזר<br>
🔹 אם היינו עושים <code>commit</code> לפני ה-rollback, לא היה דרך חזרה. לכן כותבים WHERE ובודקים תוצאה לפני commit
</p>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">חלק 2: מחיקת נתונים - DELETE</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
הפקודה <code>DELETE FROM</code> מוחקת שורות שלמות מהטבלה. גם כאן <strong>חייבים WHERE</strong>.<br>
בחברה שלנו יש עובד בן 67 שיוצא לפנסיה. נמחק כל מי שגילו מעל 65 - ושוב נסתכל לפני ואחרי:
</p>

In [ ]:
print("לפני המחיקה - עובדים מעל גיל 65:")
show('SELECT id, name, age FROM employees WHERE age > 65')

cursor.execute('SELECT COUNT(*) FROM employees')
print("מספר עובדים לפני:", cursor.fetchone()[0])

# מחיקה עם תנאי (ופרמטר)
retirement_age = 65  # שנו כאן
cursor.execute('DELETE FROM employees WHERE age > ?', (retirement_age,))
print(f"🗑️ נמחקו {cursor.rowcount} רשומות")
conn.commit()

cursor.execute('SELECT COUNT(*) FROM employees')
print("מספר עובדים אחרי:", cursor.fetchone()[0])

print("\nמי נשאר מעל גיל 65?")
show('SELECT id, name, age FROM employees WHERE age > 65')

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה כאן?</strong><br>
🔹 בדקנו קודם עם <code>SELECT</code> את מי הפקודה תמחק - זה טריק מצוין: <strong>כותבים SELECT עם אותו WHERE לפני שמוחקים</strong><br>
🔹 מספר העובדים ירד מ-12 ל-11<br>
🔹 מחיקה עם <code>commit</code> היא סופית. מי שרוצה את העובד בחזרה צריך להריץ שוב את תא "הכנת מסד הנתונים"
</p>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">חלק 3: GROUP BY - קיבוץ</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
עד עכשיו פונקציות כמו <code>COUNT</code> ו-<code>AVG</code> סיכמו את כל הטבלה לשורה אחת.<br>
עם <code>GROUP BY</code> אפשר לקבל סיכום <strong>לכל קבוצה בנפרד</strong> - למשל לכל מחלקה:<br>
🔹 <code>GROUP BY department</code> אומר "קבץ את כל העובדים של אותה מחלקה לשורה אחת"<br>
🔹 <code>AS</code> נותן שם נוח לעמודה המחושבת
</p>

In [ ]:
cursor.execute('''
    SELECT department, COUNT(*) AS num_employees, ROUND(AVG(salary)) AS avg_salary
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC
''')

print("📊 סטטיסטיקות לפי מחלקה:\n")
for department, num_employees, avg_salary in cursor.fetchall():
    print(f"{department} - מספר עובדים: {num_employees}, שכר ממוצע ₪{avg_salary:,.0f}")

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה כאן?</strong><br>
קיבלנו שורה אחת לכל מחלקה. בכל שורה יש את שם המחלקה, כמה עובדים יש בה והשכר הממוצע שלה.<br>
כלל חשוב: כל עמודה ב-<code>SELECT</code> חייבת להיות או עמודת הקיבוץ (<code>department</code>) או פונקציית צבירה (<code>COUNT</code>, <code>AVG</code>...).
</p>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">חלק 4: HAVING - סינון קבוצות</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
נניח שאנחנו רוצים רק מחלקות עם <strong>יותר מעובד אחד</strong>.<br>
אי אפשר לכתוב <code>WHERE COUNT(*) &gt; 1</code>, כי <code>WHERE</code> מסנן שורות <strong>לפני</strong> הקיבוץ, ואז עוד אין ספירה.<br>
בשביל לסנן <strong>אחרי</strong> הקיבוץ יש <code>HAVING</code>.<br><br>
סדר הפקודות: <code>WHERE</code> (מסנן שורות) ← <code>GROUP BY</code> (מקבץ) ← <code>HAVING</code> (מסנן קבוצות) ← <code>ORDER BY</code> (ממיין).
</p>

In [ ]:
cursor.execute('''
    SELECT department, COUNT(*) AS num_employees
    FROM employees
    GROUP BY department
    HAVING COUNT(*) > 1
''')
print("מחלקות עם יותר מעובד אחד:")
for department, num_employees in cursor.fetchall():
    print(f"- {department}: {num_employees} עובדים")

print()

# שילוב של WHERE ו-HAVING
cursor.execute('''
    SELECT department, ROUND(AVG(salary)) AS avg_salary
    FROM employees
    WHERE age < 40
    GROUP BY department
    HAVING AVG(salary) > 13000
    ORDER BY avg_salary DESC
''')
print("מחלקות שהשכר הממוצע של עובדיהן מתחת לגיל 40 מעל 13,000:")
for department, avg_salary in cursor.fetchall():
    print(f"- {department}: ₪{avg_salary:,.0f}")

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה כאן?</strong><br>
🔹 בדוגמה הראשונה, <code>HAVING COUNT(*) &gt; 1</code> השאיר רק מחלקות עם 2 עובדים ומעלה<br>
🔹 בדוגמה השנייה קודם <code>WHERE age &lt; 40</code> זרק עובדים מבוגרים, אחר כך חישבנו ממוצע לכל מחלקה, ורק אז <code>HAVING</code> השאיר מחלקות עם ממוצע גבוה
</p>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">חלק 5: JOIN - חיבור טבלאות</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
בטבלת העובדים יש רק את <em>שם</em> המחלקה. מי המנהל שלה ובאיזו קומה היא נמצאת? המידע הזה בטבלה אחרת (<code>departments</code>).<br>
<code>JOIN</code> מחבר שתי טבלאות לפי עמודה משותפת: <code>ON employees.department = departments.name</code>.<br>
כדי לקצר, נותנים לטבלאות שמות קצרים (alias): <code>employees AS e</code> ו-<code>departments AS d</code>.<br><br>
יש שני סוגים עיקריים:<br>
🔹 <strong>INNER JOIN</strong> - רק שורות שיש להן התאמה בשתי הטבלאות<br>
🔹 <strong>LEFT JOIN</strong> - כל השורות מהטבלה השמאלית (זו שאחרי FROM), גם אם אין להן התאמה. איפה שאין התאמה מקבלים <code>None</code>
</p>

In [ ]:
# INNER JOIN - רק עובדים שהמחלקה שלהם קיימת בטבלת המחלקות
cursor.execute('''
    SELECT e.name, e.department, d.manager, d.floor
    FROM employees AS e
    INNER JOIN departments AS d ON e.department = d.name
    ORDER BY e.id
''')
inner_rows = cursor.fetchall()

print("INNER JOIN:")
for name, department, manager, floor in inner_rows:
    print(f"{name} ({department}) - מנהל/ת: {manager}, קומה {floor}")
print(f"\nסה\"כ {len(inner_rows)} שורות")

In [ ]:
# LEFT JOIN - כל העובדים, גם אם המחלקה שלהם לא קיימת בטבלת המחלקות
cursor.execute('''
    SELECT e.name, e.department, d.manager, d.floor
    FROM employees AS e
    LEFT JOIN departments AS d ON e.department = d.name
    ORDER BY e.id
''')
left_rows = cursor.fetchall()

print("LEFT JOIN:")
for name, department, manager, floor in left_rows:
    print(f"{name} ({department}) - מנהל/ת: {manager}, קומה {floor}")
print(f"\nסה\"כ {len(left_rows)} שורות")

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה כאן?</strong><br>
🔹 ב-INNER JOIN קיבלנו 10 שורות: <strong>ליאת שרון</strong> נעלמה, כי המחלקה שלה ("מחקר") לא קיימת בטבלת המחלקות<br>
🔹 ב-LEFT JOIN קיבלנו 11 שורות: ליאת חזרה, אבל המנהל והקומה שלה הם <code>None</code><br>
🔹 הטבלה "השמאלית" היא תמיד זו שכתובה אחרי <code>FROM</code><br><br>
עכשיו נהפוך את הכיוון: נתחיל מטבלת המחלקות ונספור כמה עובדים יש בכל אחת. נשתמש ב-LEFT JOIN כדי שגם מחלקה בלי עובדים תופיע (עם 0):
</p>

In [ ]:
cursor.execute('''
    SELECT d.name, COUNT(e.id) AS num_employees
    FROM departments AS d
    LEFT JOIN employees AS e ON e.department = d.name
    GROUP BY d.name
    ORDER BY num_employees DESC
''')

print("מספר עובדים בכל מחלקה (כולל מחלקות ריקות):")
for department, num_employees in cursor.fetchall():
    print(f"- {department}: {num_employees}")

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה כאן?</strong><br>
🔹 מחלקת "רכש" מופיעה עם 0, כי LEFT JOIN שמר אותה למרות שאין לה עובדים<br>
🔹 השתמשנו ב-<code>COUNT(e.id)</code> ולא ב-<code>COUNT(*)</code>: <code>COUNT(e.id)</code> סופר רק שורות שבהן יש עובד, ולכן קיבלנו 0. <code>COUNT(*)</code> היה סופר את שורת ה"None" ונותן 1
</p>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">טעויות נפוצות</p>

<ul style="text-align: right; direction: rtl; float: right; clear: both;">
    <li><strong>שוכחים WHERE ב-UPDATE או ב-DELETE</strong> - הפקודה חלה על כל הטבלה. תמיד בדקו עם SELECT קודם</li>
    <li><strong>שוכחים commit</strong> - השינויים לא נשמרים בקובץ</li>
    <li><strong>לוקחים תנאי של קבוצה ושמים ב-WHERE</strong> - לתנאי על COUNT / AVG / SUM צריך HAVING</li>
    <li><strong>גרשיים כפולים לטקסט ב-SQL</strong> - בתוך SQL כותבים טקסט בגרשיים בודדים</li>
    <li><strong>קוראים לשאילתה אחרי close()</strong> - מקבלים Cannot operate on a closed database. מריצים שוב מתא ההכנה</li>
</ul>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">נסו בעצמכם</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/exercise.svg" style="height: 50px !important;" alt="תרגול">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>תרגול</strong>:<br>
            הוסיפו לעובדת "ליאת שרון" את המחלקה "מחקר" בטבלת המחלקות (מנהל: "ד״ר רון בר", קומה 5), ואז הריצו שוב את שאילתת ה-INNER JOIN.
כמה שורות מקבלים עכשיו? (רמז: <code>INSERT INTO departments VALUES (7, ...)</code>)
        </p>
    </div>
</div>

In [ ]:
# נסו כאן:

## <p style="text-align: right; direction: rtl; float: right; clear: both;">סגירת החיבור</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
זה התא האחרון במחברת: אחריו לא מריצים שאילתות על <code>conn</code>.<br>
(אם רוצים להמשיך להתנסות - מריצים שוב מתא "הכנת מסד הנתונים של החברה".)
</p>

In [ ]:
conn.close()
print("החיבור נסגר 👋")

## <p style="text-align: right; direction: rtl; float: right; clear: both;">סיכום</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
היום למדנו:
</p>

<ul style="text-align: right; direction: rtl; float: right; clear: both;">
    <li><code>UPDATE</code> - עדכון נתונים (ותמיד עם <code>WHERE</code>)</li>
    <li><code>DELETE</code> - מחיקה (גם כאן תמיד עם <code>WHERE</code>)</li>
    <li><code>commit</code> ו-<code>rollback</code> - שמירה וביטול של שינויים</li>
    <li><code>ORDER BY</code> ו-<code>LIMIT</code> - מיון והגבלת תוצאות</li>
    <li><code>GROUP BY</code> - קיבוץ וסטטיסטיקות לכל קבוצה</li>
    <li><code>HAVING</code> - סינון של קבוצות</li>
    <li><code>INNER JOIN</code> ו-<code>LEFT JOIN</code> - חיבור טבלאות</li>
</ul>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
במחברת הבאה נעבוד עם קובץ CSV ונשמור תוצאות לקבצים! 🚀
</p>